In [1]:
import pandas as pd

taxonomy = pd.read_csv("Train/train_taxonomy.tsv", sep="\t", names=["accession", "tax_id"])
taxonomy.head()


,accession,tax_id
0,A0A0C5B5G6,9606
1,A0JNW5,9606
2,A0JP26,9606
3,A0PK11,9606
4,A1A4S6,9606


In [2]:
import pandas as pd

# Read TSV file
train_terms = pd.read_csv('Train/train_terms.tsv', sep='\t')


# Display first few rows
train_terms.head()


,EntryID,term,aspect
0,Q5W0B1,GO:0000785,C
1,Q5W0B1,GO:0004842,F
2,Q5W0B1,GO:0051865,P
3,Q5W0B1,GO:0006275,P
4,Q5W0B1,GO:0006513,P


In [3]:
import pronto

ontology = pronto.Ontology("Train/go-basic.obo")
data = []
# .terms is now a method, so call it with ()
for term in ontology.terms():
    data.append({
        "term_id": term.id,
        "term_name": term.name
    })
ontology = pd.DataFrame(data)
ontology.head()

,term_id,term_name
0,GO:0000001,mitochondrion inheritance
1,GO:0000002,mitochondrial genome maintenance
2,GO:0000003,obsolete reproduction
3,GO:0000005,obsolete ribosomal chaperone activity
4,GO:0000006,high-affinity zinc transmembrane transporter a...


In [4]:
from Bio import SeqIO
import pandas as pd

# Read all sequences from FASTA file
sequences = []

for record in SeqIO.parse("Train/train_sequences.fasta", "fasta"):
    # Extract accession ID (middle part of sp|A0A0C5B5G6|MOTSC_HUMAN)
    accession = record.id.split('|')[1] if '|' in record.id else record.id
    
    # Extract protein name and organism
    entry_name = record.id.split('|')[2] if len(record.id.split('|')) > 2 else ''
    protein_name = entry_name.split('_')[0] if '_' in entry_name else entry_name
    organism = entry_name.split('_')[1] if '_' in entry_name else ''
    
    sequences.append({
        'accession': accession,
        'protein_name': protein_name,
        'organism': organism,
        'sequence': str(record.seq),
        'length': len(record.seq)
    })

# Convert to DataFrame
sequences_df = pd.DataFrame(sequences)

# Display first few rows
print(f"Total sequences: {len(sequences_df)}")
sequences_df.head()

Total sequences: 82404


,accession,protein_name,organism,sequence,length
0,A0A0C5B5G6,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
1,A0JNW5,BLT3B,HUMAN,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464
2,A0JP26,POTB3,HUMAN,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581
3,A0PK11,CLRN2,HUMAN,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232
4,A1A4S6,RHG10,HUMAN,MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...,786


# Check for overlap colunms 

#### taxonomy: 
    - accession_id 

#### train sequences
    - accession 

#### train terms 
    - Entry ID - can be mapped with accession
    - term (GO)

#### ontology 
    - term ID (GO) -> can be mapped with term in train terms

In [5]:
# Compare accession identifier in taxonomy, train sequences, train terms 

taxonomy_ids = set(taxonomy["accession"].unique())
sequences_ids = set(sequences_df["accession"].unique())
train_terms_ids = set(train_terms["EntryID"].unique())

In [6]:
missing_in_sequences_from_taxonomy = taxonomy_ids - sequences_ids
print(missing_in_sequences_from_taxonomy)

set()


In [7]:
missing_in_train_terms_from_taxonomy = taxonomy_ids - train_terms_ids
print(missing_in_train_terms_from_taxonomy)

set()


In [8]:
missing_in_taxonomy_from_seq = sequences_ids - taxonomy_ids
print(missing_in_taxonomy_from_seq)

set()


In [9]:
missing_in_taxonomy_from_train_terms = train_terms_ids - taxonomy_ids
print(missing_in_taxonomy_from_train_terms)

set()


In [10]:
missing_in_sequeces_from_train_terms = train_terms_ids - sequences_ids
print(missing_in_sequeces_from_train_terms)

set()


Since there is no conflict, can merge all four data into one dataframe using accession

In [11]:
merged_taxonomy_train_terms = pd.merge(taxonomy, train_terms, left_on="accession", right_on="EntryID", how='left')
merged_taxonomy_train_terms.head()

,accession,tax_id,EntryID,term,aspect
0,A0A0C5B5G6,9606,A0A0C5B5G6,GO:0001649,P
1,A0A0C5B5G6,9606,A0A0C5B5G6,GO:0033687,P
2,A0A0C5B5G6,9606,A0A0C5B5G6,GO:0005615,C
3,A0A0C5B5G6,9606,A0A0C5B5G6,GO:0005634,C
4,A0A0C5B5G6,9606,A0A0C5B5G6,GO:0005739,C


In [12]:
merged_taxonomy_train_terms.isnull().values.any()

np.False_

In [13]:
merged_taxonomy_train_terms.isnull().sum()


accession    0
tax_id       0
EntryID      0
term         0
aspect       0
dtype: int64

In [14]:
merged_taxonomy_train_terms.drop(columns=["EntryID"], inplace=True)

In [15]:
merged_taxonomy_train_terms

,accession,tax_id,term,aspect
0,A0A0C5B5G6,9606,GO:0001649,P
1,A0A0C5B5G6,9606,GO:0033687,P
2,A0A0C5B5G6,9606,GO:0005615,C
3,A0A0C5B5G6,9606,GO:0005634,C
4,A0A0C5B5G6,9606,GO:0005739,C
...,...,...,...,...
537022,Q9Y7Q3,284812,GO:0005634,C
537023,Q9Y7Q3,284812,GO:0005739,C
537024,Q9Y7Q3,284812,GO:0005829,C
537025,Q9Y816,284812,GO:0005737,C


In [16]:
merged_train_seq = pd.merge(merged_taxonomy_train_terms, sequences_df, on="accession", how="left") 
merged_train_seq

,accession,tax_id,term,aspect,protein_name,organism,sequence,length
0,A0A0C5B5G6,9606,GO:0001649,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
1,A0A0C5B5G6,9606,GO:0033687,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
2,A0A0C5B5G6,9606,GO:0005615,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
3,A0A0C5B5G6,9606,GO:0005634,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
4,A0A0C5B5G6,9606,GO:0005739,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
...,...,...,...,...,...,...,...,...
537022,Q9Y7Q3,284812,GO:0005634,C,YQ6A,SCHPO,MHSSRRKYNDMWTARLLIRSDQKEEKYPSFKKNAGKAINAHLIPKL...,149
537023,Q9Y7Q3,284812,GO:0005739,C,YQ6A,SCHPO,MHSSRRKYNDMWTARLLIRSDQKEEKYPSFKKNAGKAINAHLIPKL...,149
537024,Q9Y7Q3,284812,GO:0005829,C,YQ6A,SCHPO,MHSSRRKYNDMWTARLLIRSDQKEEKYPSFKKNAGKAINAHLIPKL...,149
537025,Q9Y816,284812,GO:0005737,C,YOND,SCHPO,MITEFIKSFLLFFFLPFFLSMPMIFATLGEFTDDQTHHYSTLPSCD...,142


In [17]:
merged_train_seq.isnull().values.any()

np.False_

In [18]:
merged_taxonomy_train_terms.isnull().sum()

accession    0
tax_id       0
term         0
aspect       0
dtype: int64

In [19]:
# Compare term identifier for ontology and merged_taxonomy_train_terms

taxonomy_terms_id = set(merged_train_seq['term'].unique())
ontology_id = set(ontology['term_id'].unique())

print(taxonomy_terms_id - ontology_id)
print(ontology_id - taxonomy_terms_id)

set()
{'GO:0031949', 'GO:0034534', 'GO:0047880', 'GO:1901091', 'GO:0045291', 'GO:0060747', 'GO:0102037', 'GO:0007590', 'GO:0046950', 'GO:1990076', 'GO:0102591', 'GO:0001315', 'GO:0010735', 'GO:0060569', 'GO:0018737', 'GO:0017092', 'GO:0102656', 'GO:0120100', 'GO:1901014', 'GO:0098768', 'GO:0010160', 'GO:0072526', 'GO:1905193', 'GO:0032284', 'GO:0046228', 'GO:0052783', 'GO:1904342', 'GO:0005907', 'GO:0048398', 'GO:0010056', 'GO:0090390', 'GO:0045181', 'GO:0042763', 'GO:0102939', 'GO:1900804', 'GO:0002303', 'GO:0046411', 'GO:0075339', 'GO:0010433', 'GO:0160108', 'GO:0050781', 'GO:0034562', 'GO:0160250', 'GO:0000194', 'GO:0030821', 'GO:0061989', 'GO:0017032', 'GO:0098881', 'GO:0050106', 'GO:0060189', 'GO:1900884', 'GO:0000335', 'GO:0031817', 'GO:1901769', 'GO:0047635', 'GO:1900544', 'GO:0036452', 'GO:0061183', 'GO:0050943', 'GO:0005660', 'GO:0034411', 'GO:0045507', 'GO:1902401', 'GO:0140525', 'GO:0110033', 'GO:0120057', 'GO:0030933', 'GO:0000191', 'GO:0014805', 'GO:0075528', 'GO:0031042',

In [20]:
go_not_taxonomy = ontology_id - taxonomy_terms_id
differences = len(go_not_taxonomy)
print(differences)

21976


There are `21,976` go terms that are not included in the train taxonomy

In [21]:
merged_taxonomy_train_terms.to_csv("merged_taxonomy_train_terms.csv", index=False)

finding: all entry id and accession id are matched

In [22]:
# Couting if there is multiple EntryID -> has many GO: Terms

count_multi = merged_train_seq.groupby("accession")['term'].count().reset_index(name='go_term_count')
multi_go = count_multi[count_multi["go_term_count"] > 1]

multi_go

,accession,go_term_count
5,A0A023FFD0,2
9,A0A023I7E1,2
12,A0A024RBG1,2
14,A0A026W182,7
15,A0A044RE18,3
...,...,...
82399,X2JI34,2
82400,X4Y2L4,2
82401,X5JA13,8
82402,X5JB51,8


In [23]:
!pip install tabulate


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from tabulate import tabulate

organism_counts = (
    merged_train_seq['organism']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Organism', 'organism': 'Count'})
)

print(tabulate(organism_counts, headers='keys', tablefmt='psql'))


+------+---------+---------+
|      | Count   |   count |
|------+---------+---------|
|    0 | HUMAN   |  155400 |
|    1 | MOUSE   |  109671 |
|    2 | ARATH   |   57405 |
|    3 | RAT     |   43418 |
|    4 | YEAST   |   37056 |
|    5 | DROME   |   28050 |
|    6 | SCHPO   |   20089 |
|    7 | CAEEL   |   14984 |
|    8 | ECOLI   |   14808 |
|    9 | DICDI   |    7301 |
|   10 | DANRE   |    6710 |
|   11 | MYCTU   |    4769 |
|   12 | CANAL   |    3304 |
|   13 | ORYSJ   |    3189 |
|   14 | CHICK   |    2594 |
|   15 | BOVIN   |    2587 |
|   16 | XENLA   |    2445 |
|   17 | PSEAE   |    1136 |
|   18 | EMENI   |     980 |
|   19 | BACSU   |     845 |
|   20 | PIG     |     833 |
|   21 | PLAF7   |     717 |
|   22 | RABIT   |     680 |
|   23 | MAIZE   |     670 |
|   24 | ASPFU   |     456 |
|   25 | CANLF   |     437 |
|   26 | TRYB2   |     406 |
|   27 | CHLRE   |     364 |
|   28 | SALTY   |     345 |
|   29 | SOLLC   |     320 |
|   30 | GIAIC   |     296 |
|   31 | ONCMY

In [25]:
merged_train_seq.head()

,accession,tax_id,term,aspect,protein_name,organism,sequence,length
0,A0A0C5B5G6,9606,GO:0001649,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
1,A0A0C5B5G6,9606,GO:0033687,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
2,A0A0C5B5G6,9606,GO:0005615,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
3,A0A0C5B5G6,9606,GO:0005634,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
4,A0A0C5B5G6,9606,GO:0005739,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16


67k+ data has multiple protein functionality (go:term)

In [26]:
merged_train_seq.to_csv("merged_final_seq.csv", index=False)

### Load the test data

In [27]:
print("\n Loading Test Data")

test_sequences = []

for record in SeqIO.parse("Test/testsuperset.fasta", "fasta"):
    parts = record.description.split()
    accession = parts[0]
    tax_id = parts[1] if len(parts) > 1 else ''

    test_sequences.append({
        'accession' : accession, 
        'tax_id' : int(tax_id),
        'sequence' : str(record.seq),
        'length' : len(record.seq)
    })

df_test_fasta = pd.DataFrame(test_sequences)
print(f"Loaded {len(df_test_fasta)} test fasta data")


 Loading Test Data
Loaded 224309 test fasta data


In [28]:
df_test_fasta.head()

,accession,tax_id,sequence,length
0,A0A0C5B5G6,9606,MRWQEMGYIFYPRKLR,16
1,A0A1B0GTW7,9606,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788
2,A0JNW5,9606,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464
3,A0JP26,9606,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581
4,A0PK11,9606,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232


In [29]:
df_test_tsv = pd.read_csv('Test/testsuperset-taxon-list.tsv', sep='\t')
df_test_tsv.head()

,ID,Species
0,9606,Homo sapiens
1,10116,Rattus norvegicus
2,39947,Oryza sativa subsp. japonica
3,7955,Danio rerio
4,7227,Drosophila melanogaster


In [30]:
df_test = pd.merge(df_test_tsv, df_test_fasta, left_on="ID", right_on="tax_id", how="left")
df_test.head()

,ID,Species,accession,tax_id,sequence,length
0,9606,Homo sapiens,A0A0C5B5G6,9606,MRWQEMGYIFYPRKLR,16
1,9606,Homo sapiens,A0A1B0GTW7,9606,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788
2,9606,Homo sapiens,A0JNW5,9606,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464
3,9606,Homo sapiens,A0JP26,9606,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581
4,9606,Homo sapiens,A0PK11,9606,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232


In [31]:
print(df_test['ID'].isna().sum())
print(df_test['tax_id'].isna().sum())


0
0


In [32]:
from tabulate import tabulate

species_counts = (
    df_test['Species']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Species', 'organism': 'Count'})
)

print(tabulate(species_counts, headers='keys', tablefmt='psql'))


+------+-----------------------------------------------------------------------------------------------------------------------------------------+---------+
|      | Species                                                                                                                                 |   count |
|------+-----------------------------------------------------------------------------------------------------------------------------------------+---------|
|    0 | Homo sapiens                                                                                                                            |   20420 |
|    1 | Mus musculus                                                                                                                            |   17240 |
|    2 | Arabidopsis thaliana                                                                                                                    |   16397 |
|    3 | Rattus norvegicus                                